# 🏦 Retail Banking Product & Fees Assistant
### Hybrid RAG with Intent Routing, Grounded Citations & Refusal Mechanism

**Project:** Capstone — GenAI Banking Assistant  
**Stack:** FAISS + BM25 + RRF | Groq LLM | LLM-as-Judge Evaluator

---
**Pipeline Overview:**
```
Query → Intent Router → Filtered Retrieval (Dense + Sparse + RRF)
      → Confidence Check → Generator (with citations) → LLM-as-Judge Eval
```

## 📦 Section 1: Install Dependencies

In [1]:
!pip install -q faiss-cpu rank_bm25 sentence-transformers groq langchain langchain-community tiktoken

^C


## 🔑 Section 2: API Key Setup

In [4]:
import os
# from google.colab import userdata

# Store your key in Colab Secrets (key icon in left sidebar) as GROQ_API_KEY
os.environ["GROQ_API_KEY"] = "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"

from groq import Groq
client = Groq(api_key=os.environ["GROQ_API_KEY"])
print("✅ Groq client initialized")

✅ Groq client initialized


## 🏗️ Section 3: Synthetic Data Generation

We generate 3 document types that mirror what a real bank's knowledge base would contain:
- **Product Terms** — account features, limits, interest rates
- **Fee Schedule** — charges per transaction/service
- **Eligibility Rules** — who qualifies for what product

In [13]:
import json
import time

def generate_doc(prompt: str, doc_name: str) -> str:
    """Generate synthetic banking document using Groq."""
    print(f"⏳ Generating {doc_name}...")
    response = client.chat.completions.create(
        model="groq/compound-mini",
        messages=[{
            "role": "user",
            "content": prompt
        }],
        temperature=0.3,
        max_tokens=2000
    )
    text = response.choices[0].message.content
    print(f"✅ {doc_name} generated ({len(text)} chars)")
    return text

In [14]:
# --- Prompts for each document type ---

PRODUCT_TERMS_PROMPT = """
Generate detailed product terms for a fictional Indian retail bank called 'NovaBank'.
Include the following products with realistic specifications:

1. NovaSavings Account - standard savings
2. NovaPrime Savings Account - premium tier
3. NovaFixed Deposit (FD) - various tenures
4. NovaRecurring Deposit (RD)
5. NovaPersonal Loan
6. NovaCreditCard - two variants (Classic and Platinum)

For each product include: interest rates, tenure options, withdrawal rules,
lock-in periods, key features, and important terms and conditions.
Format clearly with product headers and bullet points.
Make values realistic for India (interest rates, INR amounts, etc.).
"""

FEE_SCHEDULE_PROMPT = """
Generate a comprehensive fee schedule for a fictional Indian retail bank called 'NovaBankk'.
Include fees for:

1. NovaSavings Account: minimum balance penalty, ATM withdrawal (own/other bank), 
   NEFT/RTGS/IMPS charges, cheque book, DD, account statement, SMS alerts
2. NovaPrime Savings Account: same categories (different/waived fees)
3. NovaFixed Deposit: premature withdrawal penalty, duplicate certificate
4. NovaPersonal Loan: processing fee, prepayment charges, late payment penalty, 
   EMI bounce charges
5. NovaCreditCard Classic: annual fee, joining fee, late payment, overlimit, 
   cash advance, forex markup, reward redemption
6. NovaCreditCard Platinum: same categories (premium tier)
7. General: Locker charges, KYC update, NACH mandate

Present as a structured table-like format with exact INR amounts.
Make it realistic for an Indian bank in 2024.
"""

ELIGIBILITY_PROMPT = """
Generate detailed eligibility criteria for a fictional Indian retail bank 'NovaBank'.
Include eligibility rules for:

1. NovaSavings Account: age, KYC docs, initial deposit, resident/NRI
2. NovaPrime Savings Account: minimum monthly balance, salary/income proof
3. NovaFixed Deposit: minimum amount, age (minor accounts?), joint accounts
4. NovaRecurring Deposit: minimum monthly installment, tenure options, age
5. NovaPersonal Loan: age range, minimum income (salaried vs self-employed),
   credit score cutoff, employment stability, existing EMI obligations (FOIR)
6. NovaCreditCard Classic: age, income, credit score, employment type
7. NovaCreditCard Platinum: higher income bar, credit score, existing relationship

Also include: documents required for each product.
Be specific with numbers (CIBIL scores, income amounts in INR, age limits).
"""

# Generate all 3 documents
product_terms_doc = generate_doc(PRODUCT_TERMS_PROMPT, "Product Terms")
time.sleep(2)  # rate limit buffer
fee_schedule_doc = generate_doc(FEE_SCHEDULE_PROMPT, "Fee Schedule")
time.sleep(2)
eligibility_doc = generate_doc(ELIGIBILITY_PROMPT, "Eligibility Rules")

⏳ Generating Product Terms...
✅ Product Terms generated (9801 chars)
⏳ Generating Fee Schedule...
✅ Fee Schedule generated (6742 chars)
⏳ Generating Eligibility Rules...
✅ Eligibility Rules generated (10552 chars)


In [ ]:
import os

os.makedirs("data/raw", exist_ok=True)

docs_map = {
    "data/raw/product_terms.txt": product_terms_doc,
    "data/raw/fee_schedule.txt": fee_schedule_doc,
    "data/raw/eligibility_rules.txt": eligibility_doc
}

for path, content in docs_map.items():
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)

print("✅ All documents saved to data/raw/")

# Preview
print("\n--- Product Terms (first 500 chars) ---")
print(product_terms_doc[:500])

✅ All documents saved to data/raw/

--- Product Terms (first 500 chars) ---
**NovaBank – Retail‑Banking Product Terms (Fictional)**  
*All rates and terms are indicative and subject to change at NovaBank’s discretion, subject to RBI regulations and the customer’s credit profile.*

---

## 1. NovaSavings Account – Standard Savings  

| Feature | Details |
|---------|---------|
| **Interest Rate** | **3.00 % p.a.** on balances up to ₹2 Lakh  <br> **3.50 % p.a.** on the portion above ₹2 Lakh (slab‑wise) |
| **Interest Calculation** | Daily on closing balance, credited **qu


## 📊 Section 4: Generate Evaluation QA Set

We generate 60 Q&A pairs — 20 per doc type — including some **unanswerable questions** to test the refusal mechanism.

In [ ]:
def generate_qa_set(doc_content: str, doc_type: str, n: int = 20) -> list:
    """Generate Q&A pairs grounded in the provided document."""
    prompt = f"""
You are creating an evaluation dataset for a banking RAG system.

Based on this banking document about {doc_type}:
---
{doc_content[:3000]}
---

Generate {n} question-answer pairs. Mix of:
- 14 answerable questions (specific, grounded in the document)
- 6 unanswerable questions (things NOT covered in the document, e.g. stock tips, forex trading, insurance)

Return ONLY valid JSON array, no extra text:
[
  {{"question": "...", "answer": "...", "answerable": true, "doc_type": "{doc_type}"}},
  {{"question": "...", "answer": "Not covered in banking documents", "answerable": false, "doc_type": "{doc_type}"}}
]
"""
    response = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.4,
        max_tokens=3000
    )
    raw = response.choices[0].message.content
    # Clean up common JSON issues
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        print(f"⚠️ JSON parse error for {doc_type}, returning empty list")
        return []

In [ ]:
print("⏳ Generating evaluation QA set...")
qa_product = generate_qa_set(product_terms_doc, "product_terms", n=20)
time.sleep(2)
qa_fees = generate_qa_set(fee_schedule_doc, "fees", n=20)
time.sleep(2)
qa_eligibility = generate_qa_set(eligibility_doc, "eligibility", n=20)

all_qa = qa_product + qa_fees + qa_eligibility

os.makedirs("data/eval", exist_ok=True)
with open("data/eval/qa_set.json", "w") as f:
    json.dump(all_qa, f, indent=2)

answerable = sum(1 for q in all_qa if q.get("answerable", True))
print(f"\n✅ QA set saved: {len(all_qa)} total | {answerable} answerable | {len(all_qa)-answerable} unanswerable")

## 🔍 Section 5: Document Chunking & Ingestion

In [ ]:
from langchain.tools import RecursiveCharacterTextSplitter
from dataclasses import dataclass, field
from typing import List, Dict

@dataclass
class Chunk:
    chunk_id: str
    text: str
    doc_type: str   # 'product_terms' | 'fees' | 'eligibility'
    source_file: str

def load_and_chunk_documents() -> List[Chunk]:
    """Load raw docs and split into chunks with metadata."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=80,
        separators=["\n\n", "\n", ". ", " "]
    )

    doc_config = [
        ("data/raw/product_terms.txt", "product_terms"),
        ("data/raw/fee_schedule.txt", "fees"),
        ("data/raw/eligibility_rules.txt", "eligibility")
    ]

    all_chunks = []
    for filepath, doc_type in doc_config:
        with open(filepath, "r") as f:
            text = f.read()

        splits = splitter.split_text(text)
        for i, chunk_text in enumerate(splits):
            all_chunks.append(Chunk(
                chunk_id=f"{doc_type}_{i:03d}",
                text=chunk_text.strip(),
                doc_type=doc_type,
                source_file=filepath
            ))
        print(f"✅ {doc_type}: {len(splits)} chunks")

    return all_chunks

chunks = load_and_chunk_documents()
print(f"\n📦 Total chunks: {len(chunks)}")

ModuleNotFoundError: No module named 'langchain.text_splitter'

## 🧠 Section 6: Build Dense Index (FAISS + BGE Embeddings)

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("⏳ Loading BGE embedding model...")
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
print("✅ Embedding model loaded")

def build_dense_index(chunks: List[Chunk]):
    """Encode chunks and build FAISS index."""
    texts = [c.text for c in chunks]
    print(f"⏳ Encoding {len(texts)} chunks...")

    # BGE requires a query prefix for retrieval
    embeddings = embed_model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True
    ).astype(np.float32)

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # Inner product (cosine after normalization)
    index.add(embeddings)

    print(f"✅ FAISS index built: {index.ntotal} vectors, dim={dim}")
    return index, embeddings

faiss_index, chunk_embeddings = build_dense_index(chunks)

## 📝 Section 7: Build Sparse Index (BM25)

In [ ]:
from rank_bm25 import BM25Okapi
import re

def tokenize(text: str) -> List[str]:
    """Simple tokenizer for BM25."""
    text = text.lower()
    tokens = re.findall(r'\b[a-z0-9]+\b', text)
    return tokens

corpus_tokens = [tokenize(c.text) for c in chunks]
bm25 = BM25Okapi(corpus_tokens)

print(f"✅ BM25 index built over {len(corpus_tokens)} documents")

## 🔀 Section 8: Hybrid Retrieval with Reciprocal Rank Fusion (RRF)

In [ ]:
from typing import Optional

def dense_search(query: str, top_k: int = 20, doc_type_filter: Optional[str] = None) -> List[tuple]:
    """Dense retrieval with optional doc_type filter. Returns (chunk_idx, score) list."""
    # BGE query prefix
    query_vec = embed_model.encode(
        [f"Represent this sentence for searching relevant passages: {query}"],
        normalize_embeddings=True
    ).astype(np.float32)

    scores, indices = faiss_index.search(query_vec, top_k * 3)  # over-fetch then filter
    results = []
    for idx, score in zip(indices[0], scores[0]):
        if idx == -1:
            continue
        if doc_type_filter and chunks[idx].doc_type != doc_type_filter:
            continue
        results.append((idx, float(score)))
        if len(results) == top_k:
            break
    return results


def sparse_search(query: str, top_k: int = 20, doc_type_filter: Optional[str] = None) -> List[tuple]:
    """BM25 retrieval with optional doc_type filter. Returns (chunk_idx, score) list."""
    query_tokens = tokenize(query)
    scores = bm25.get_scores(query_tokens)

    # Get ranked indices
    ranked_indices = np.argsort(scores)[::-1]
    results = []
    for idx in ranked_indices:
        if doc_type_filter and chunks[idx].doc_type != doc_type_filter:
            continue
        results.append((idx, float(scores[idx])))
        if len(results) == top_k:
            break
    return results


def reciprocal_rank_fusion(dense_results: List[tuple], sparse_results: List[tuple],
                           k: int = 60, top_n: int = 5) -> List[tuple]:
    """
    RRF: score(d) = sum(1 / (k + rank_i))
    Returns top_n (chunk_idx, rrf_score) pairs.
    """
    rrf_scores: Dict[int, float] = {}

    for rank, (idx, _) in enumerate(dense_results):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k + rank + 1)

    for rank, (idx, _) in enumerate(sparse_results):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k + rank + 1)

    sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_results[:top_n]


def hybrid_retrieve(query: str, top_n: int = 5,
                    doc_type_filter: Optional[str] = None) -> List[Chunk]:
    """Full hybrid retrieval pipeline."""
    dense_res = dense_search(query, top_k=20, doc_type_filter=doc_type_filter)
    sparse_res = sparse_search(query, top_k=20, doc_type_filter=doc_type_filter)
    fused = reciprocal_rank_fusion(dense_res, sparse_res, top_n=top_n)
    return [chunks[idx] for idx, _ in fused], [score for _, score in fused]


# Quick test
test_chunks, test_scores = hybrid_retrieve("What is the minimum balance for savings account?")
print("\n🔍 Hybrid Retrieval Test:")
for chunk, score in zip(test_chunks, test_scores):
    print(f"  [{chunk.doc_type}] (rrf={score:.4f}) {chunk.text[:120]}...")

## 🎯 Section 9: Intent Router

Routes the query to the relevant document namespace before retrieval — reduces noise and improves precision.

In [ ]:
INTENT_SYSTEM_PROMPT = """
You are an intent classifier for a banking assistant.
Classify the user query into ONE of these intents:

- fees: Questions about charges, penalties, costs, service fees, transaction fees
- eligibility: Questions about who can apply, requirements, documents, criteria, qualifications
- product_terms: Questions about features, interest rates, tenure, limits, how products work
- out_of_scope: Questions unrelated to banking products (stocks, insurance, crypto, general advice)

Respond with ONLY one word: fees | eligibility | product_terms | out_of_scope
"""

def classify_intent(query: str) -> str:
    """Classify query intent using LLM."""
    response = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[
            {"role": "system", "content": INTENT_SYSTEM_PROMPT},
            {"role": "user", "content": query}
        ],
        temperature=0.0,
        max_tokens=10
    )
    intent = response.choices[0].message.content.strip().lower()
    valid_intents = {"fees", "eligibility", "product_terms", "out_of_scope"}
    return intent if intent in valid_intents else "product_terms"  # default fallback


# Test intent routing
test_queries = [
    "What is the NEFT charge for savings account?",
    "What is the minimum CIBIL score for personal loan?",
    "How does the fixed deposit interest work?",
    "Should I invest in Nifty 50 or Bitcoin?"
]

print("🎯 Intent Routing Tests:")
for q in test_queries:
    intent = classify_intent(q)
    print(f"  '{q[:60]}' → {intent}")
    time.sleep(0.5)

## 🛡️ Section 10: Confidence Check & Refusal Mechanism

Two-layer refusal:
1. **Intent-level**: out_of_scope → immediate refusal
2. **Retrieval-level**: top RRF score below threshold → low-confidence refusal

In [ ]:
CONFIDENCE_THRESHOLD = 0.007  # RRF score threshold (tune based on your data)

def check_retrieval_confidence(scores: List[float]) -> bool:
    """Returns True if retrieval is confident enough to answer."""
    if not scores:
        return False
    return scores[0] >= CONFIDENCE_THRESHOLD


REFUSAL_MESSAGE = (
    "I'm sorry, I don't have reliable information to answer this question. "
    "Please contact your branch or our 24/7 helpline at 1800-XXX-XXXX for assistance."
)

OUT_OF_SCOPE_MESSAGE = (
    "This question is outside the scope of NovaBank's product and fee information. "
    "I can only assist with questions about our savings accounts, fixed deposits, "
    "personal loans, credit cards, fees, and eligibility criteria."
)

print("✅ Refusal mechanism configured")
print(f"   Confidence threshold: {CONFIDENCE_THRESHOLD}")

## 💬 Section 11: Generator with Grounded Citations

In [ ]:
GENERATOR_SYSTEM_PROMPT = """
You are a helpful and accurate banking assistant for NovaBank.
Answer the customer's question using ONLY the provided context chunks.

Rules:
1. Ground every factual claim in the context. Cite the source as [Source: chunk_id].
2. If the context doesn't contain enough information, say: 
   "Based on available information, I cannot fully answer this. Please contact our branch."
3. Be concise and professional. Use bullet points for lists of fees/features.
4. Do NOT make up numbers, rates, or policies not present in the context.
"""

def generate_answer(query: str, retrieved_chunks: List[Chunk]) -> str:
    """Generate cited answer from retrieved context."""
    context_str = "\n\n".join([
        f"[Source: {c.chunk_id}]\n{c.text}"
        for c in retrieved_chunks
    ])

    user_prompt = f"""Context:
{context_str}

Customer Question: {query}

Answer (cite sources as [Source: chunk_id]):"""

    response = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[
            {"role": "system", "content": GENERATOR_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=600
    )
    return response.choices[0].message.content.strip()

print("✅ Generator configured")

## 🔄 Section 12: Full RAG Pipeline

Putting it all together: Intent → Retrieve → Confidence Check → Generate

In [ ]:
import time as time_module

def rag_pipeline(query: str, top_n: int = 5, verbose: bool = True) -> dict:
    """
    Full RAG pipeline.
    Returns: dict with answer, intent, retrieved_chunks, latency, refused flag
    """
    start = time_module.time()

    # Step 1: Intent Classification
    intent = classify_intent(query)
    if verbose:
        print(f"  🎯 Intent: {intent}")

    # Step 2: Out-of-scope refusal
    if intent == "out_of_scope":
        latency = time_module.time() - start
        return {
            "query": query,
            "answer": OUT_OF_SCOPE_MESSAGE,
            "intent": intent,
            "retrieved_chunks": [],
            "rrf_scores": [],
            "refused": True,
            "refusal_reason": "out_of_scope",
            "latency_ms": round(latency * 1000, 2)
        }

    # Step 3: Hybrid Retrieval (filtered by intent)
    retrieved, scores = hybrid_retrieve(query, top_n=top_n, doc_type_filter=intent)
    if verbose:
        print(f"  🔍 Retrieved {len(retrieved)} chunks (top score: {scores[0]:.4f if scores else 0})")

    # Step 4: Confidence check
    if not check_retrieval_confidence(scores):
        latency = time_module.time() - start
        return {
            "query": query,
            "answer": REFUSAL_MESSAGE,
            "intent": intent,
            "retrieved_chunks": retrieved,
            "rrf_scores": scores,
            "refused": True,
            "refusal_reason": "low_confidence",
            "latency_ms": round((time_module.time() - start) * 1000, 2)
        }

    # Step 5: Generate answer with citations
    answer = generate_answer(query, retrieved)
    latency = time_module.time() - start

    return {
        "query": query,
        "answer": answer,
        "intent": intent,
        "retrieved_chunks": retrieved,
        "rrf_scores": scores,
        "refused": False,
        "refusal_reason": None,
        "latency_ms": round(latency * 1000, 2)
    }


# ---- DEMO RUNS ----
demo_queries = [
    "What are the NEFT charges for NovaSavings account?",
    "What is the minimum CIBIL score required for NovaPersonal Loan?",
    "Tell me about NovaFixed Deposit interest rates and tenure options.",
    "Should I buy gold or invest in mutual funds?"  # out of scope
]

for q in demo_queries:
    print(f"\n{'='*60}")
    print(f"❓ Query: {q}")
    result = rag_pipeline(q, verbose=True)
    print(f"  🛡️ Refused: {result['refused']} {('(' + result['refusal_reason'] + ')') if result['refused'] else ''}")
    print(f"  ⏱️ Latency: {result['latency_ms']}ms")
    print(f"  💬 Answer:\n{result['answer']}")
    time.sleep(1)

## ⚖️ Section 13: LLM-as-Judge Evaluator

Evaluates three metrics:
- **Faithfulness** — is the answer grounded in the retrieved context?
- **Answer Relevancy** — does the answer address the question?
- **Refusal Correctness** — did the system refuse when it should have (and not refuse when it shouldn't)?

In [ ]:
FAITHFULNESS_PROMPT = """
You are evaluating whether an AI answer is faithful to the provided context.

Context:
{context}

Question: {question}
Answer: {answer}

Evaluate faithfulness: Does the answer ONLY contain claims supported by the context?
Score 0.0 to 1.0 where:
- 1.0 = all claims are grounded in context
- 0.5 = some claims unsupported
- 0.0 = answer contradicts or ignores context

Respond ONLY with a JSON object: {{"score": <float>, "reason": "<one sentence>"}}
"""

RELEVANCY_PROMPT = """
You are evaluating whether an AI answer addresses the user's question.

Question: {question}
Answer: {answer}

Score answer relevancy 0.0 to 1.0 where:
- 1.0 = directly and completely answers the question
- 0.5 = partially answers or goes off-topic
- 0.0 = completely misses the question

Respond ONLY with a JSON object: {{"score": <float>, "reason": "<one sentence>"}}
"""


def llm_judge(prompt_template: str, **kwargs) -> dict:
    """Run LLM judge with given prompt template."""
    prompt = prompt_template.format(**kwargs)
    response = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=150
    )
    raw = response.choices[0].message.content.strip()
    try:
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        return json.loads(raw)
    except:
        return {"score": 0.0, "reason": "parse error"}


def evaluate_result(result: dict, ground_truth_answer: str = None,
                    should_be_refused: bool = False) -> dict:
    """Evaluate a single RAG result across all metrics."""
    metrics = {}

    # Refusal correctness
    if should_be_refused:
        metrics["refusal_correctness"] = 1.0 if result["refused"] else 0.0
    else:
        metrics["refusal_correctness"] = 0.0 if result["refused"] else 1.0

    if result["refused"]:
        metrics["faithfulness"] = None  # N/A for refused
        metrics["relevancy"] = None
        return metrics

    # Context for faithfulness
    context = "\n".join([c.text for c in result["retrieved_chunks"]])

    faith = llm_judge(FAITHFULNESS_PROMPT,
                      context=context,
                      question=result["query"],
                      answer=result["answer"])
    metrics["faithfulness"] = faith.get("score", 0.0)
    metrics["faithfulness_reason"] = faith.get("reason", "")

    time.sleep(0.5)

    rel = llm_judge(RELEVANCY_PROMPT,
                    question=result["query"],
                    answer=result["answer"])
    metrics["relevancy"] = rel.get("score", 0.0)
    metrics["relevancy_reason"] = rel.get("reason", "")

    return metrics


print("✅ LLM-as-Judge evaluator configured")

## 📈 Section 14: Full Evaluation Run on QA Set

In [ ]:
# Load QA set
with open("data/eval/qa_set.json", "r") as f:
    qa_set = json.load(f)

# Run evaluation on a sample (full run = slow; use all 60 for final eval)
EVAL_SAMPLE_SIZE = 20  # Change to len(qa_set) for full eval
eval_sample = qa_set[:EVAL_SAMPLE_SIZE]

print(f"\n📊 Running evaluation on {EVAL_SAMPLE_SIZE} samples...")
print("(This will take a few minutes due to API calls)\n")

eval_results = []
latencies = []

for i, qa in enumerate(eval_sample):
    print(f"  [{i+1}/{EVAL_SAMPLE_SIZE}] {qa['question'][:60]}...")

    # Run pipeline
    result = rag_pipeline(qa["question"], verbose=False)
    latencies.append(result["latency_ms"])

    # Evaluate
    should_refuse = not qa.get("answerable", True)
    metrics = evaluate_result(result, should_be_refused=should_refuse)

    eval_results.append({
        "question": qa["question"],
        "expected_answer": qa.get("answer", ""),
        "generated_answer": result["answer"],
        "intent": result["intent"],
        "refused": result["refused"],
        "should_refuse": should_refuse,
        "latency_ms": result["latency_ms"],
        **metrics
    })

    time.sleep(1)  # Rate limit buffer

print("\n✅ Evaluation complete!")

## 📊 Section 15: Metrics Summary

In [ ]:
import statistics

# Filter answered vs refused
answered = [r for r in eval_results if not r["refused"]]
refused_results = [r for r in eval_results if r["refused"]]

# Faithfulness
faith_scores = [r["faithfulness"] for r in answered if r.get("faithfulness") is not None]
avg_faithfulness = statistics.mean(faith_scores) if faith_scores else 0

# Relevancy
rel_scores = [r["relevancy"] for r in answered if r.get("relevancy") is not None]
avg_relevancy = statistics.mean(rel_scores) if rel_scores else 0

# Refusal correctness
refusal_correct = [r for r in eval_results if r["refusal_correctness"] == 1.0]
refusal_correctness = len(refusal_correct) / len(eval_results) if eval_results else 0

# Latency
sorted_latencies = sorted(latencies)
p95_latency = sorted_latencies[int(len(sorted_latencies) * 0.95)] if sorted_latencies else 0
avg_latency = statistics.mean(latencies) if latencies else 0

print("\n" + "="*50)
print("📊 EVALUATION RESULTS")
print("="*50)
print(f"  Samples evaluated    : {len(eval_results)}")
print(f"  Answered             : {len(answered)}")
print(f"  Refused              : {len(refused_results)}")
print(f"")
print(f"  Faithfulness         : {avg_faithfulness:.3f}")
print(f"  Answer Relevancy     : {avg_relevancy:.3f}")
print(f"  Refusal Correctness  : {refusal_correctness:.3f}")
print(f"")
print(f"  Avg Latency          : {avg_latency:.0f}ms")
print(f"  p95 Latency          : {p95_latency:.0f}ms")
print("="*50)

# Save results
with open("data/eval/eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2, default=str)

summary = {
    "samples": len(eval_results),
    "faithfulness": round(avg_faithfulness, 3),
    "answer_relevancy": round(avg_relevancy, 3),
    "refusal_correctness": round(refusal_correctness, 3),
    "avg_latency_ms": round(avg_latency, 1),
    "p95_latency_ms": round(p95_latency, 1)
}
with open("data/eval/summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n✅ Results saved to data/eval/")

## 🔬 Section 16: Error Analysis

In [ ]:
# Low faithfulness cases
low_faith = [
    r for r in answered
    if r.get("faithfulness") is not None and r["faithfulness"] < 0.7
]

# Wrong refusals
wrong_refusals = [r for r in eval_results if r["refusal_correctness"] == 0.0]

print(f"⚠️  Low Faithfulness Cases (< 0.7): {len(low_faith)}")
for r in low_faith[:3]:
    print(f"  Q: {r['question'][:80]}")
    print(f"  Faith: {r['faithfulness']:.2f} | {r.get('faithfulness_reason', '')}")
    print()

print(f"\n⚠️  Wrong Refusal Decisions: {len(wrong_refusals)}")
for r in wrong_refusals[:3]:
    refused_when = "should NOT have" if not r["should_refuse"] else "should have"
    print(f"  Q: {r['question'][:80]}")
    print(f"  System refused={r['refused']} but {refused_when} refused")
    print()

## 🎮 Section 17: Interactive Demo

In [ ]:
def interactive_query(query: str):
    """Pretty-print a single query through the full pipeline."""
    print("\n" + "="*60)
    print(f"❓ Query: {query}")
    print("-"*60)

    result = rag_pipeline(query, verbose=True)

    print(f"\n💬 Answer:")
    print(result["answer"])

    if result["retrieved_chunks"] and not result["refused"]:
        print(f"\n📎 Sources used:")
        for chunk in result["retrieved_chunks"]:
            print(f"  • [{chunk.chunk_id}] ({chunk.doc_type}): {chunk.text[:80]}...")

    print(f"\n⏱️ Latency: {result['latency_ms']}ms")
    print("="*60)


# Try your own queries!
interactive_query("What documents do I need to apply for a NovaPersonal Loan?")

In [ ]:
# Try more queries
interactive_query("What is the annual fee for NovaCreditCard Platinum?")

In [ ]:
interactive_query("How do I invest in the stock market?")  # Should be refused

---
## ✅ Summary & Next Steps

### What this notebook covers:
| Component | Implementation |
|---|---|
| Data | Synthetic banking docs (LLM-generated) |
| Chunking | RecursiveCharacterTextSplitter (500 tokens, 80 overlap) |
| Dense Retrieval | FAISS + BGE-small-en-v1.5 |
| Sparse Retrieval | BM25 (rank_bm25) |
| Fusion | Reciprocal Rank Fusion (k=60) |
| Intent Routing | LLM classifier → doc namespace filter |
| Refusal | Out-of-scope + low-confidence threshold |
| Citations | Source chunk IDs in every answer |
| Evaluation | LLM-as-Judge (faithfulness + relevancy + refusal correctness + p95 latency) |

### Next steps to convert to project structure:
1. Move each section into `src/` modules
2. Add FastAPI wrapper (`/query` endpoint)
3. Swap FAISS → Qdrant for production
4. Add Streamlit UI for demo
5. Add LangGraph state machine for multi-turn conversations